# Phase 10 — CI-CD

**Objectifs:**
- [x] Git Integration
- [x] GitHub Actions
- [x] Pytest
- [x] Code Validation
- [x] Déploiement

**Livrable:** CI/CD Pipeline

**CI (Continuous Integration):** à chaque modification, on vérifie **automatiquement** que le code est propre et que les tests passent.
**CD (Continuous Deployment):** si tout est vert sur `main`, on **déploie automatiquement** le Job sur Databricks.

```
git push ──► GitHub Actions ──► ruff (lint) ──► validation notebooks ──► pytest ──► bundle validate ──► (main) bundle deploy
```

## 0. Setup

In [1]:
import subprocess, os
import lakehouse_config
lakehouse_config._add_cli_to_path()   # rend la CLI databricks disponible


def sh(cmd):
    """Execute une commande shell a la racine du projet et affiche la sortie."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, encoding="utf-8", errors="replace")
    out = "\n".join(l for l in (r.stdout + r.stderr).splitlines() if "aitools" not in l)
    print(out.strip()[-4000:])
    print(f"-> code retour: {r.returncode}")
    return r.returncode

## 1. Git Integration

Structure du dépôt:
```
Lakehouse-Databricks/
├── Phase 01 … Phase 11 *.ipynb     # notebooks (un par phase)
├── lakehouse_config.py             # configuration partagee (tables, connexion)
├── lakehouse_transforms.py         # logique Silver + framework Data Quality
├── tests/                          # tests pytest
├── resources/lakehouse_job.yml     # definition du Job (Phase 9)
├── databricks.yml                  # Databricks Asset Bundle
├── .github/workflows/ci.yml        # pipeline CI/CD
├── pyproject.toml                  # config ruff + pytest
├── requirements*.txt               # dependances
└── README.md
```

**Côté Databricks:** *Workspace → Create → Git folder* → URL du dépôt GitHub → les notebooks sont synchronisés (pull / commit / push depuis l'UI).

**Workflow Git recommandé:** une branche par fonctionnalité → Pull Request → la CI s'exécute → merge sur `main` → déploiement automatique.

In [2]:
sh("git status --short")
sh("git log --oneline -5")

M .gitignore
 M "Phase 01 \342\200\224 D\303\251couverte du Dataset.ipynb"
 M "Phase 02 \342\200\224 Spark Fundamentals.ipynb"
 M "Phase 03 \342\200\224 Bronze Ingestion.ipynb"
 M "Phase 04 \342\200\224 Silver Transformation.ipynb"
 M "Phase 05 \342\200\224 Gold Analytics.ipynb"
 M "Phase 06 \342\200\224 Data Quality.ipynb"
 M "Phase 07 \342\200\224 Incremental Processing.ipynb"
 M "Phase 08 \342\200\224 Unity Catalog.ipynb"
 M "Phase 09 \342\200\224 Orchestration.ipynb"
 M "Phase 10 \342\200\224 CI-CD.ipynb"
 M "Phase 11 \342\200\224 Finalisation.ipynb"
 M README.md
 M __pycache__/lakehouse_config.cpython-312.pyc
 M databricks.yml
 M lakehouse_config.py
?? .github/
?? docs/
?? lakehouse_transforms.py
?? pyproject.toml
?? requirements-dev.txt
?? requirements.txt
?? resources/
?? tests/
-> code retour: 0
088dc4c 01
e7d677a first commit
-> code retour: 0


0

In [3]:
print(open(".gitignore", encoding="utf-8").read())


.databricks
.venv/
data/
__pycache__/
.pytest_cache/
.ruff_cache/



## 2. GitHub Actions
Le workflow `.github/workflows/ci.yml`:

In [4]:
print(open(".github/workflows/ci.yml", encoding="utf-8").read())

# Phase 10 — CI/CD
# - Pull request / push : lint (ruff) + validation des notebooks + tests (pytest) + validation du bundle
# - Push sur main      : deploiement du bundle (Job Databricks) sur la cible dev
#
# Secrets GitHub requis (Settings > Secrets and variables > Actions) :
#   DATABRICKS_HOST   ex: https://dbc-xxxx.cloud.databricks.com
#   DATABRICKS_TOKEN  Personal Access Token Databricks

name: CI-CD

on:
  pull_request:
  push:
    branches: [main]
  workflow_dispatch:

env:
  DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
  DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}

jobs:
  validate:
    name: Lint, tests et validation
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
          cache: pip

      - name: Installer les dependances
        run: pip install -r requirements-dev.txt

      - name: Lint (ruff)
        run: ruff check .

      - name: Valider les noteboo

**Secrets à configurer** dans GitHub (*Settings → Secrets and variables → Actions*):

| Secret | Valeur |
|---|---|
| `DATABRICKS_HOST` | `https://dbc-8e6d5a22-8ef2.cloud.databricks.com` |
| `DATABRICKS_TOKEN` | Personal Access Token (*Databricks → Settings → Developer → Access tokens*) |

Sans ces secrets, la CI exécute quand même le lint et les tests sans Spark (les tests Spark sont automatiquement **skippés**).

## 3. Pytest
- `tests/test_config.py`: tests **sans Spark** (structure des tables, règles DQ, snake_case).
- `tests/test_transforms.py`: tests **avec Spark** (règles de rejet, nulls, `trip_id`, zones, DQ) sur de petits DataFrames.
- `tests/conftest.py`: fixture `spark` partagée (skip si pas d'authentification).

In [5]:
print(open("tests/test_transforms.py", encoding="utf-8").read()[:2500])

"""Tests des transformations sur de petits DataFrames (necessite Databricks Connect)."""
from datetime import datetime
from decimal import Decimal

import pytest

from lakehouse_transforms import (
    DataQualityError,
    add_reject_reason,
    add_trip_id,
    assert_no_critical_failures,
    clean_zones,
    evaluate_rules,
    handle_nulls,
)

pytestmark = pytest.mark.spark

T0, T1 = datetime(2024, 1, 15, 10, 0), datetime(2024, 1, 15, 10, 20)
TRIP_SCHEMA = ("pickup_datetime timestamp, dropoff_datetime timestamp, trip_distance double, "
               "fare_amount decimal(10,2), total_amount decimal(10,2), pu_location_id int, do_location_id int")


def reasons(spark, row):
    r = add_reject_reason(spark.createDataFrame([row], TRIP_SCHEMA)).collect()[0].reject_reason
    return set(r.split("; ")) if r else set()


def test_valid_trip_has_no_reject_reason(spark):
    assert reasons(spark, (T0, T1, 2.5, Decimal("12.00"), Decimal("15.00"), 10, 20)) == set()


@pytest.mark.parametrize(

In [6]:
sh(".venv\\Scripts\\python.exe -m pytest -v")

============================= test session starts =============================
platform win32 -- Python 3.12.15, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\marou\OneDrive\Desktop\Lakehouse-Databricks\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\marou\OneDrive\Desktop\Lakehouse-Databricks
configfile: pyproject.toml
testpaths: tests
plugins: platformdirs-4.14.5
collecting ... collected 17 items

tests/test_config.py::test_tables_are_fully_qualified PASSED             [  5%]
tests/test_config.py::test_medallion_schemas PASSED                      [ 11%]
tests/test_config.py::test_column_mapping_is_snake_case PASSED           [ 17%]
tests/test_config.py::test_dq_rules_are_well_formed PASSED               [ 23%]
tests/test_config.py::test_reject_reasons_unique PASSED                  [ 29%]
tests/test_transforms.py::test_valid_trip_has_no_reject_reason PASSED    [ 35%]
tests/test_transforms.py::test_reject_reasons[row0-expected0] PASSED     [ 41%]
tests/test_transforms.py

0

## 4. Code Validation
### 4.1 Lint avec ruff (style, imports, bugs probables)

In [7]:
sh(".venv\\Scripts\\ruff.exe check . --statistics")


-> code retour: 0


0

### 4.2 Validation des notebooks (format nbformat valide)

In [8]:
import glob, nbformat
for p in sorted(glob.glob("Phase *.ipynb")):
    nb = nbformat.read(p, as_version=4)
    nbformat.validate(nb)
    errors = sum(1 for c in nb.cells if c.cell_type == "code" for o in c.get("outputs", []) if o.output_type == "error")
    print(f"OK  {p:45s} {len(nb.cells):3d} cellules | erreurs dans les sorties: {errors}")

OK  Phase 01 — Découverte du Dataset.ipynb         46 cellules | erreurs dans les sorties: 0
OK  Phase 02 — Spark Fundamentals.ipynb            52 cellules | erreurs dans les sorties: 0
OK  Phase 03 — Bronze Ingestion.ipynb              21 cellules | erreurs dans les sorties: 0
OK  Phase 04 — Silver Transformation.ipynb         23 cellules | erreurs dans les sorties: 0
OK  Phase 05 — Gold Analytics.ipynb                23 cellules | erreurs dans les sorties: 0
OK  Phase 06 — Data Quality.ipynb                  20 cellules | erreurs dans les sorties: 0
OK  Phase 07 — Incremental Processing.ipynb        20 cellules | erreurs dans les sorties: 0
OK  Phase 08 — Unity Catalog.ipynb                 22 cellules | erreurs dans les sorties: 0
OK  Phase 09 — Orchestration.ipynb                 18 cellules | erreurs dans les sorties: 0
OK  Phase 10 — CI-CD.ipynb                         19 cellules | erreurs dans les sorties: 0
OK  Phase 11 — Finalisation.ipynb                  14 cellules | erreu

### 4.3 Validation du bundle Databricks

In [9]:
sh('databricks bundle validate -t dev -p "marouane mrf"')

Name: Lakehouse-Databricks
Target: dev
Workspace:
  Host: https://dbc-8e6d5a22-8ef2.cloud.databricks.com
  User: marwane2001morfi@gmail.com
  Path: /Workspace/Users/marwane2001morfi@gmail.com/.bundle/Lakehouse-Databricks/dev

Validation OK!
-> code retour: 0


0

## 5. Déploiement

- **Automatique**: job `deploy` du workflow, uniquement sur **push vers `main`** et seulement si `validate` a réussi (`needs: validate`).
- **Manuel** (depuis le poste):
```bash
databricks bundle deploy -t dev -p "marouane mrf"
```
- **Environnement prod** (évolution): ajouter une cible `prod` dans `databricks.yml` (`mode: production`, autre workspace ou autre catalog) et un déploiement déclenché par un **tag** de version (`v1.0.0`).

## Conclusion — CI/CD Pipeline
- Code versionné avec **Git**, logique métier isolée dans des **modules Python testables**.
- **17 tests pytest** (5 sans Spark, 12 avec Spark) + lint **ruff** + validation des **notebooks** et du **bundle**.
- **GitHub Actions**: validation à chaque PR/push, **déploiement automatique** du Job sur `main`.